# 05 - Class 1 Reserved Slot Strategy

This notebook compares three booking rules under one editable in-notebook scenario:

- pooled FCFS baseline with no protected slots
- strict Class 1 reservation, where protected slots cannot be used by Class 2
- released Class 1 reservation, where Class 1 gets first pass and Class 2 can backfill unused protected slots

Edit the scenario cell below to change capacity, horizon, demand, behavior rules, reserved slots, or seeds. The notebook does not read or write YAML config files.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "simulation" / "engine.py").exists() and (
            candidate / "analysis" / "metrics.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import PatientClassParams, SimulationConfig, ThresholdRule

plt.style.use("default")


## Editable Scenario Config

Change values in `SCENARIO` to test a different setup. Everything is built in memory from this cell.

In [ ]:
SCENARIO = {
    "slots_per_day": 32,
    "horizon_days": 14,
    "burn_in_days": 30,
    "measure_days": 365,
    "cooldown_days": 10,
    "reserved_class_id": 1,
    "reserved_slots_per_day": 10,
    "seeds": range(5101, 5131),
    "classes": {
        1: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
        2: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
    },
}

scenario_preview = pd.Series(
    {key: value for key, value in SCENARIO.items() if key not in {"classes", "seeds"}},
    name="value",
).to_frame()
class_preview = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "lambda_per_day": params["lambda_per_day"],
            "cancel_prob": params["cancel_prob"],
            "balk_threshold": params["balk_prob"]["threshold"],
            "balk_low": params["balk_prob"]["low"],
            "balk_high": params["balk_prob"]["high"],
            "no_show_threshold": params["no_show_prob"]["threshold"],
            "no_show_low": params["no_show_prob"]["low"],
            "no_show_high": params["no_show_prob"]["high"],
        }
        for class_id, params in SCENARIO["classes"].items()
    ]
)

display(scenario_preview)
display(class_preview)


## Policy Setup

The pooled policy uses one shared capacity pool. The strict and released policies split each appointment day into a protected pool and a general pool.

In [ ]:
def build_config(scenario: dict, *, reserved: bool, release_reserved_slots: bool, seed: int | None = None) -> SimulationConfig:
    classes = {
        int(class_id): PatientClassParams(
            class_id=int(class_id),
            lambda_per_day=params["lambda_per_day"],
            balk_prob=ThresholdRule(**params["balk_prob"]),
            cancel_prob=params["cancel_prob"],
            no_show_prob=ThresholdRule(**params["no_show_prob"]),
            value=params.get("value", 1.0),
        )
        for class_id, params in scenario["classes"].items()
    }
    return SimulationConfig(
        slots_per_day=scenario["slots_per_day"],
        horizon_days=scenario["horizon_days"],
        burn_in_days=scenario["burn_in_days"],
        measure_days=scenario["measure_days"],
        cooldown_days=scenario["cooldown_days"],
        classes=classes,
        seed=seed,
        reserved_class_id=scenario["reserved_class_id"] if reserved else None,
        reserved_slots_per_day=scenario["reserved_slots_per_day"] if reserved else 0,
        release_reserved_slots=release_reserved_slots if reserved else False,
    )


policy_configs = {
    "Pooled FCFS": build_config(SCENARIO, reserved=False, release_reserved_slots=False),
    "Strict C1 reservation": build_config(SCENARIO, reserved=True, release_reserved_slots=False),
    "Released C1 reservation": build_config(SCENARIO, reserved=True, release_reserved_slots=True),
}

policy_table = pd.DataFrame(
    [
        {
            "policy": name,
            "reserved_class_id": cfg.reserved_class_id,
            "reserved_slots_per_day": cfg.reserved_slots_per_day,
            "general_slots_per_day": cfg.slots_per_day - cfg.reserved_slots_per_day,
            "release_reserved_slots": cfg.release_reserved_slots,
        }
        for name, cfg in policy_configs.items()
    ]
)

policy_table


## Multi-Seed Runs

Each policy is run over the same seed list from `SCENARIO`. The only difference between policies is the booking rule.

In [ ]:
aggregate_rows = []
class_rows = []

for policy, config in policy_configs.items():
    for seed in tqdm(list(SCENARIO["seeds"]), desc=policy):
        result = ClinicAppointmentSimulation(replace(config, seed=seed)).run()
        fixed = {"policy": policy, "seed": seed}
        aggregate_rows.append(aggregate_result_row(result, fixed))
        class_rows.extend(class_result_rows(result, fixed))

aggregate_df = pd.DataFrame(aggregate_rows)
class_df = pd.DataFrame(class_rows)

for outcome in ["balked", "no_offer", "canceled", "no_show", "unresolved_booked"]:
    aggregate_df[f"{outcome}_rate"] = (
        aggregate_df[f"total_{outcome}"] / aggregate_df["total_arrivals"]
    )

aggregate_df["served_rate"] = aggregate_df["overall_percent_serviced"]
aggregate_df.head()


## Aggregate Comparison

These metrics show the access-capacity tradeoff. Strict reservation can protect Class 1 capacity, but it can leave protected slots unused. Released reservation tests whether Class 2 backfill recovers that capacity.

In [ ]:
aggregate_metrics = [
    "average_utilization",
    "served_rate",
    "mean_offered_booking_delay",
    "mean_accepted_booking_delay",
    "balked_rate",
    "no_offer_rate",
    "canceled_rate",
    "no_show_rate",
]

aggregate_summary = (
    aggregate_df.groupby("policy")[aggregate_metrics]
    .agg(["mean", "std"])
    .reindex(policy_configs.keys())
    .round(3)
)

aggregate_summary


In [ ]:
plot_metrics = [
    ("average_utilization", "Average utilization"),
    ("served_rate", "Served rate"),
    ("no_offer_rate", "No-offer rate"),
    ("mean_offered_booking_delay", "Mean offered delay"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (metric, title) in zip(axes.flat, plot_metrics):
    means = aggregate_df.groupby("policy")[metric].mean().reindex(policy_configs.keys())
    stds = aggregate_df.groupby("policy")[metric].std().reindex(policy_configs.keys())
    means.plot(kind="bar", yerr=stds, capsize=3, ax=ax, color=["#6b7280", "#2563eb", "#059669"])
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)

fig.tight_layout()


## Class-Level Effects

The reservation policy is designed around Class 1, so class-level served rate, delay, no-offer, and balking are the main diagnostics.

In [ ]:
class_df["no_offer_rate"] = (class_df["no_offer"] / class_df["arrivals"]).fillna(0)
class_df["balking_rate"] = (class_df["balked"] / class_df["offered"]).fillna(0)

class_metrics = [
    "percent_serviced",
    "mean_offered_booking_delay",
    "no_offer_rate",
    "balking_rate",
]

class_summary = (
    class_df.groupby(["policy", "class_id"])[class_metrics]
    .mean()
    .reindex(policy_configs.keys(), level="policy")
    .round(3)
)

class_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

served_by_class = (
    class_df.groupby(["policy", "class_id"])["percent_serviced"]
    .mean()
    .unstack("class_id")
    .reindex(policy_configs.keys())
)
delay_by_class = (
    class_df.groupby(["policy", "class_id"])["mean_offered_booking_delay"]
    .mean()
    .unstack("class_id")
    .reindex(policy_configs.keys())
)

served_by_class.plot(kind="bar", ax=axes[0], color=["#2563eb", "#dc2626"])
axes[0].set_title("Served rate by class")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", alpha=0.25)

delay_by_class.plot(kind="bar", ax=axes[1], color=["#2563eb", "#dc2626"])
axes[1].set_title("Mean offered delay by class")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)

fig.tight_layout()


## Reading The Result

Use strict reservation to measure the cost of true protection: Class 2 cannot touch protected slots, so unused protected capacity can lower utilization or increase Class 2 no-offer.

Use released reservation to measure the recovery value of flexible protection: Class 1 gets first pass on protected slots, but Class 2 can backfill capacity that Class 1 did not use.